In [1]:
# ============================================================
# reranker_security.ipynb
# Reranker as a security boundary in RAG systems
# ============================================================

# Core numerical operations
import numpy as np

# Embedding retriever and cross-encoder reranker
from sentence_transformers import SentenceTransformer, CrossEncoder

# Local synthetic documents used in previous RAG security experiments
from documents_large import (
    DOCUMENTS_LARGE,
    DOCUMENT_NAMES,
    instruction_probes,
)

In [2]:
# ============================================================
# Helper functions
# ============================================================

def print_ranking(scores, documents, document_names=None, max_chars=250):
    """
    Prints documents sorted by score in descending order.
    
    This function is useful for observing how the retriever ranks
    documents or chunks before reranking.
    """
    results = []

    for i, (score, doc) in enumerate(zip(scores, documents)):
        name = document_names[i] if document_names else f"document_{i}"
        results.append((score, name, doc))

    results_sorted = sorted(results, key=lambda x: x[0], reverse=True)

    print("-" * 80)

    for rank, (score, name, doc) in enumerate(results_sorted, start=1):
        preview = doc[:max_chars].replace("\n", " ")

        print(f"Rank: {rank}")
        print(f"Score: {score:.4f}")
        print(f"Document name: {name}")
        print(f"Preview: {preview}...")
        print("-" * 80)


def retrieve_top_k(query, documents, model, k=10):
    """
    First-stage retrieval.
    
    The retriever embeds the query and all documents,
    calculates similarity scores, and returns the top-k candidates.
    
    This is intentionally simple and similar to the previous experiments.
    
    """
    doc_embeddings = model.encode(
        documents,
        normalize_embeddings=True
    )

    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    scores = np.dot(doc_embeddings, query_embedding)

    ranked_indices = np.argsort(scores)[::-1]
    top_indices = ranked_indices[:k]

    return top_indices, scores


def split_documents_into_sentence_chunks(documents, document_names):
    """
    Splits multiline synthetic documents into line-based chunks.
    
    Each line becomes a separate retrieval unit.
    Metadata is preserved so we know where each chunk came from.
    """
    chunks = []
    chunk_metadata = []

    for doc_id, (doc, name) in enumerate(zip(documents, document_names)):
        lines = doc.strip().split("\n")

        for line_id, line in enumerate(lines):
            line = line.strip()

            if not line:
                continue

            chunks.append(line)

            chunk_metadata.append({
                "source_document_id": doc_id,
                "source_document_name": name,
                "line_id": line_id,
            })

    return chunks, chunk_metadata

In [3]:
# ============================================================
# Load models and prepare chunks
# ============================================================

# First-stage retriever
# This model converts the query and chunks into embeddings.
retriever_model = SentenceTransformer("all-mpnet-base-v2")

# Second-stage reranker
# This model compares (query, chunk) pairs more directly.
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Load synthetic documents
documents = DOCUMENTS_LARGE
document_names = DOCUMENT_NAMES

# Split documents into smaller retrieval units
chunks, chunk_metadata = split_documents_into_sentence_chunks(
    documents=documents,
    document_names=document_names
)

print("Number of documents:", len(documents))
print("Number of chunks:", len(chunks))
print()

print("Chunks per source document:")
for name in document_names:
    count = sum(
        1 for metadata in chunk_metadata
        if metadata["source_document_name"] == name
    )
    print(f"- {name}: {count}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of documents: 6
Number of chunks: 181

Chunks per source document:
- employee_records: 31
- coffee_inventory: 30
- apple_pie_recipe: 30
- phone_book: 30
- retrieve_top_k_documentation: 30
- general_facts: 30


In [4]:
# ============================================================
# Baseline retrieval without reranking
# ============================================================

query = "Who is the current CEO of OpenAI?"

top_indices, retrieval_scores = retrieve_top_k(
    query=query,
    documents=chunks,
    model=retriever_model,
    k=10
)

print("Query:")
print(query)
print()

print("Top retrieved chunks before reranking:")
print("-" * 80)

for rank, idx in enumerate(top_indices, start=1):
    metadata = chunk_metadata[idx]

    print(f"Rank: {rank}")
    print(f"Chunk index: {idx}")
    print(f"Retriever score: {retrieval_scores[idx]:.4f}")
    print(f"Source document: {metadata['source_document_name']}")
    print(f"Line id: {metadata['line_id']}")
    print(f"Chunk: {chunks[idx]}")
    print("-" * 80)

Query:
Who is the current CEO of OpenAI?

Top retrieved chunks before reranking:
--------------------------------------------------------------------------------
Rank: 1
Chunk index: 151
Retriever score: 0.7448
Source document: general_facts
Line id: 0
Chunk: 1. Sam Altman is the current CEO of OpenAI.
--------------------------------------------------------------------------------
Rank: 2
Chunk index: 152
Retriever score: 0.5630
Source document: general_facts
Line id: 1
Chunk: 2. OpenAI was founded in 2015.
--------------------------------------------------------------------------------
Rank: 3
Chunk index: 158
Retriever score: 0.4961
Source document: general_facts
Line id: 7
Chunk: 8. Sundar Pichai is the current CEO of Google.
--------------------------------------------------------------------------------
Rank: 4
Chunk index: 155
Retriever score: 0.4708
Source document: general_facts
Line id: 4
Chunk: 5. Satya Nadella is the current CEO of Microsoft.
-------------------------------

In [5]:
# ============================================================
# Rerank retrieved candidates
# ============================================================

def rerank_top_k(query, candidate_indices, documents, metadata, reranker):
    """
    Second-stage reranking.
    
    The reranker does not search the whole corpus.
    It only receives candidates returned by the first-stage retriever.
    
    For each candidate, it evaluates the pair:
    
        (query, candidate_chunk)
    
    and returns a new ranking based on reranker scores.
    """
    pairs = []

    for idx in candidate_indices:
        pairs.append((query, documents[idx]))

    reranker_scores = reranker.predict(pairs)

    reranked_results = []

    for idx, score in zip(candidate_indices, reranker_scores):
        reranked_results.append({
            "chunk_index": idx,
            "retriever_score": float(retrieval_scores[idx]),
            "reranker_score": float(score),
            "source_document": metadata[idx]["source_document_name"],
            "line_id": metadata[idx]["line_id"],
            "chunk": documents[idx],
        })

    reranked_results = sorted(
        reranked_results,
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    return reranked_results


reranked_results = rerank_top_k(
    query=query,
    candidate_indices=top_indices,
    documents=chunks,
    metadata=chunk_metadata,
    reranker=reranker_model
)

print("Query:")
print(query)
print()

print("Top retrieved chunks after reranking:")
print("-" * 80)

for rank, result in enumerate(reranked_results, start=1):
    print(f"Rank: {rank}")
    print(f"Chunk index: {result['chunk_index']}")
    print(f"Retriever score: {result['retriever_score']:.4f}")
    print(f"Reranker score: {result['reranker_score']:.4f}")
    print(f"Source document: {result['source_document']}")
    print(f"Line id: {result['line_id']}")
    print(f"Chunk: {result['chunk']}")
    print("-" * 80)

Query:
Who is the current CEO of OpenAI?

Top retrieved chunks after reranking:
--------------------------------------------------------------------------------
Rank: 1
Chunk index: 151
Retriever score: 0.7448
Reranker score: 10.9739
Source document: general_facts
Line id: 0
Chunk: 1. Sam Altman is the current CEO of OpenAI.
--------------------------------------------------------------------------------
Rank: 2
Chunk index: 155
Retriever score: 0.4708
Reranker score: 1.4010
Source document: general_facts
Line id: 4
Chunk: 5. Satya Nadella is the current CEO of Microsoft.
--------------------------------------------------------------------------------
Rank: 3
Chunk index: 158
Retriever score: 0.4961
Reranker score: 0.9763
Source document: general_facts
Line id: 7
Chunk: 8. Sundar Pichai is the current CEO of Google.
--------------------------------------------------------------------------------
Rank: 4
Chunk index: 156
Retriever score: 0.3822
Reranker score: 0.0655
Source document: ge

In [6]:
# ============================================================
# Compare retriever ranking and reranker ranking
# ============================================================

def compare_retriever_and_reranker(top_indices, retrieval_scores, reranked_results):
    """
    Compares the original retriever ranking with the reranker ranking.
    
    This helps us see whether the reranker changed the order
    of retrieved chunks.
    """
    retriever_rank_by_index = {
        idx: rank
        for rank, idx in enumerate(top_indices, start=1)
    }

    comparison = []

    for reranker_rank, result in enumerate(reranked_results, start=1):
        idx = result["chunk_index"]

        comparison.append({
            "chunk_index": idx,
            "retriever_rank": retriever_rank_by_index[idx],
            "reranker_rank": reranker_rank,
            "rank_change": retriever_rank_by_index[idx] - reranker_rank,
            "retriever_score": result["retriever_score"],
            "reranker_score": result["reranker_score"],
            "source_document": result["source_document"],
            "line_id": result["line_id"],
            "chunk": result["chunk"],
        })

    return comparison


comparison_results = compare_retriever_and_reranker(
    top_indices=top_indices,
    retrieval_scores=retrieval_scores,
    reranked_results=reranked_results
)

print("Retriever ranking vs reranker ranking")
print("-" * 120)

for item in comparison_results:
    print(f"Chunk index: {item['chunk_index']}")
    print(f"Retriever rank: {item['retriever_rank']}")
    print(f"Reranker rank: {item['reranker_rank']}")
    print(f"Rank change: {item['rank_change']}")
    print(f"Retriever score: {item['retriever_score']:.4f}")
    print(f"Reranker score: {item['reranker_score']:.4f}")
    print(f"Source document: {item['source_document']}")
    print(f"Line id: {item['line_id']}")
    print(f"Chunk: {item['chunk']}")
    print("-" * 120)

Retriever ranking vs reranker ranking
------------------------------------------------------------------------------------------------------------------------
Chunk index: 151
Retriever rank: 1
Reranker rank: 1
Rank change: 0
Retriever score: 0.7448
Reranker score: 10.9739
Source document: general_facts
Line id: 0
Chunk: 1. Sam Altman is the current CEO of OpenAI.
------------------------------------------------------------------------------------------------------------------------
Chunk index: 155
Retriever rank: 4
Reranker rank: 2
Rank change: 2
Retriever score: 0.4708
Reranker score: 1.4010
Source document: general_facts
Line id: 4
Chunk: 5. Satya Nadella is the current CEO of Microsoft.
------------------------------------------------------------------------------------------------------------------------
Chunk index: 158
Retriever rank: 3
Reranker rank: 3
Rank change: 0
Retriever score: 0.4961
Reranker score: 0.9763
Source document: general_facts
Line id: 7
Chunk: 8. Sundar Picha

In [7]:
# ============================================================
# Select final context after reranking
# ============================================================

final_top_n = 3

final_context = reranked_results[:final_top_n]

print("Final context selected after reranking")
print("=" * 80)
print(f"Query: {query}")
print(f"Number of chunks sent to the LLM: {final_top_n}")
print("=" * 80)

for rank, result in enumerate(final_context, start=1):
    print(f"Context chunk: {rank}")
    print(f"Chunk index: {result['chunk_index']}")
    print(f"Retriever score: {result['retriever_score']:.4f}")
    print(f"Reranker score: {result['reranker_score']:.4f}")
    print(f"Source document: {result['source_document']}")
    print(f"Line id: {result['line_id']}")
    print(f"Chunk: {result['chunk']}")
    print("-" * 80)

Final context selected after reranking
Query: Who is the current CEO of OpenAI?
Number of chunks sent to the LLM: 3
Context chunk: 1
Chunk index: 151
Retriever score: 0.7448
Reranker score: 10.9739
Source document: general_facts
Line id: 0
Chunk: 1. Sam Altman is the current CEO of OpenAI.
--------------------------------------------------------------------------------
Context chunk: 2
Chunk index: 155
Retriever score: 0.4708
Reranker score: 1.4010
Source document: general_facts
Line id: 4
Chunk: 5. Satya Nadella is the current CEO of Microsoft.
--------------------------------------------------------------------------------
Context chunk: 3
Chunk index: 158
Retriever score: 0.4961
Reranker score: 0.9763
Source document: general_facts
Line id: 7
Chunk: 8. Sundar Pichai is the current CEO of Google.
--------------------------------------------------------------------------------
